In [1]:
import numpy as np
import qutip as qt
import scipy.sparse as sp
import matplotlib.pyplot as plt

# Import custom modules
from quantum_gates import GATE_DICTIONARY
from circuit_engine import apply_instruction
from generator import generate_random_circuit
from visualize import draw_ascii_circuit

print("All modules loaded successfully! Ready to test.")

All modules loaded successfully! Ready to test.


In [2]:
# Cell 2: Generate a test circuit
num_qubits = 4
num_layers = 6

print(f"--- Generating a {num_qubits}-Qubit Circuit with {num_layers} Layers ---\n")

# Call your generator (Make sure it returns ops, descs, AND instructions)
ops, descs, instructions = generate_random_circuit(num_qubits, num_layers)

# Print out the English descriptions of each layer
for i, desc in enumerate(descs):
    print(f"Layer {i+1}: {desc}")

# ---------------------------------------------------------
# check EVERY layer for shape and unitarity
# ---------------------------------------------------------
for i, op in enumerate(ops):
    # This will crash the program immediately if any layer is the wrong shape
    expected_shape = (2**num_qubits, 2**num_qubits)
    assert op.shape == expected_shape, f"Error: Layer {i+1} shape is {op.shape}, expected {expected_shape}"
    
    # This will crash the program if any layer is not unitary
    assert op.isunitary, f"Error: Layer {i+1} is not unitary!"

print("\nSuccess! All layers are the correct shape and perfectly unitary.")

--- Generating a 4-Qubit Circuit with 6 Layers ---

Layer 1: 1-qubit layer: H(0), H(1), Y(2), H(3)
Layer 2: 2-qubit layer: C-H(control=3, target=0)
Layer 3: 1-qubit layer: S(0), H(1), S(2), H(3)
Layer 4: 2-qubit layer: C-X(control=3, target=2)
Layer 5: 1-qubit layer: Z(0), T(1), Z(2), H(3)
Layer 6: 1-qubit layer: Y(0), Z(1), Z(2), T(3)

Success! All layers are the correct shape and perfectly unitary.


In [3]:
# Draw the circuit
draw_ascii_circuit(num_qubits, instructions)


=== Generated Quantum Circuit ===
q0: ───[H]────[H]────[S]───────────[Z]────[Y]───
q1: ───[H]─────│─────[H]───────────[T]────[Z]───
q2: ───[Y]─────│─────[S]────[X]────[Z]────[Z]───
q3: ───[H]─────■─────[H]─────■─────[H]────[T]───



In [4]:
# Apply the circuit to a state
# Create the initial state |0000>
ket0 = qt.basis(2, 0)
initial_state = qt.tensor([ket0 for _ in range(num_qubits)])

print("Initial State (First 5 amplitudes):")
print(initial_state.full().flatten()[:5]) 

# apply the circuit using Dense matrices
current_state = initial_state.full()
for layer_op in ops:
    current_state = layer_op.full() @ current_state

print("\nFinal State Vector after Circuit (First 5 amplitudes):")
print(np.round(current_state.flatten()[:5], 3))

# Verify probabilities sum to 1
total_probability = np.sum(np.abs(current_state)**2)
print(f"\nTotal Probability: {total_probability:.5f} (Should be exactly 1.0)")

Initial State (First 5 amplitudes):
[1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]

Final State Vector after Circuit (First 5 amplitudes):
[1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]

Total Probability: 1.00000 (Should be exactly 1.0)
